In [14]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

In [15]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")

In [16]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [17]:
load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant")

In [18]:
class State(TypedDict):
    topic: str # user input
    plan: Plan
    sections: Annotated[List[str], operator.add] # reducer: results from workers get concatenated into this list
    final: str

In [19]:
def orchestrator(state: State) -> dict:

    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(content="Create a blog plan with 5-7 sections on the following topic."),
            HumanMessage(content=f"Topic: {state['topic']}")
        ]
    )

    return {"plan": plan}